# 01 - First circuit and load flow

## Objective

Build a source-line-load circuit and compare the load-bus voltage returned by direct OpenDSS with the same typed Case run through the CEPT public CLI.

## Source, assumptions, and units

The source is the checked-in `public/examples/first_circuit_case.json` definition: a three-phase, 12.47 kV source and load bus joined by `line1`, length 1.0 km, with the declared positive- and zero-sequence ohmic data. The load is 100.0 kW at power factor 0.95. These are demonstrator inputs, not measurements. Voltage magnitude is line-to-neutral per unit (`pu`); line length is km and load power is kW.

## Prediction

The load-bus voltage should be below the 1.0 pu source voltage because the feeder has nonzero impedance and the load consumes real and reactive power. Direct OpenDSS and CEPT should agree within a declared teaching tolerance because they use the same source assumptions.

## Action

First write the exact example input into this notebook's working directory, then solve it directly in OpenDSS. The CEPT action is a streamed subprocess call to `cept study run`; its output is not hidden behind a Python API.

## Verification

The CLI's exact run directory contains `case.json`, `results.json`, `manifest.json`, `validation_report.json`, and `public-verification.json`. `cept study verify` must return literal `passed: true`. The comparison uses solver-returned values from the direct calculation and `results.json`.

## Interpretation

The difference is a translation/regression check for this matched demonstrator. It is not a project acceptance tolerance and does not replace review of source data.

## Exercise

Change exactly one declared input in `CASE_PAYLOAD`, predict the direction of the voltage change, restart the kernel, and rerun every cell. Keep the changed input explicit; do not tune an output to meet the tolerance.

## Runtime requirements

Use Python 3.10 or newer with an existing installed `cept` command, or provide a caller-owned wheel through `CEPT_WHEEL_URL` and its exact `CEPT_WHEEL_SHA256`. The wheel must provide the public CEPT CLI and OpenDSS runtime. No released PyPI version is assumed. Jupyter is needed only to execute the notebook.

In [1]:
# @title Setup — run once, then read the results below
import urllib.request, hashlib
_HELPER_URL = "https://raw.githubusercontent.com/sarutesri/cept-studio-edu/75b4f394aa096a604123e6e1739d2581e8ef9476/public/notebooks/_lesson.py"
_HELPER_SHA256 = "8618face0c62b85127ffadc7b662bc177ab3ba5a36e32a09ee5223f562e02ba2"
_blob = urllib.request.urlopen(_HELPER_URL, timeout=60).read()
assert hashlib.sha256(_blob).hexdigest() == _HELPER_SHA256, "lesson helper hash mismatch"
exec(compile(_blob, "lesson helper", "exec"))


CEPT_WHEEL_URL not supplied; using the existing installed environment.
CLI: cept --version


cept-power-studio 0.2.0.dev0
lesson helpers ready: cli/read/table/cards + WORKSPACE.


In [2]:
# @title Inputs — demonstrator values (no need to edit)
CASE_PATH = WORKSPACE / 'first_circuit_case.json'
CASE_PAYLOAD = first_circuit_case()
CASE_PATH.write_text(json.dumps(CASE_PAYLOAD, indent=2) + '\n', encoding='utf-8')
print('Source input:', CASE_PATH)
table(['declared input', 'value', 'unit'], [('frequency', 60, 'Hz'), ('line length', 1.0, 'km'), ('load', 100.0, 'kW'), ('load power factor', 0.95, '1')])


Source input: <notebook-workspace>\first_circuit_case.json
| declared input | value | unit |
| --- | --- | --- |
| frequency | 60 | Hz |
| line length | 1.0 | km |
| load | 100.0 | kW |
| load power factor | 0.95 | 1 |


In [3]:
import opendssdirect as dss

for command in [
    'Clear',
    'New Circuit.first basekv=12.47 pu=1.0 phases=3 bus1=source',
    'New Line.line1 bus1=source.1.2.3 bus2=load.1.2.3 phases=3 length=1 units=km r1=0.2 x1=0.4 r0=0.6 x0=1.2 c1=0 c0=0',
    'New Load.load1 bus1=load.1.2.3 phases=3 conn=wye kv=12.47 kw=100 pf=0.95',
    'CalcVoltageBases',
    'Solve',
]:
    dss.Text.Command(command)
assert dss.Solution.Converged()
dss.Circuit.SetActiveBus('load')
direct_values = dss.Bus.puVmagAngle()
direct_by_phase = {phase: float(direct_values[2 * (phase - 1)]) for phase in (1, 2, 3)}
table(['source', 'phase', 'voltage magnitude', 'unit'], [('direct OpenDSS', phase, direct_by_phase[phase], 'pu') for phase in (1, 2, 3)])
assert all(value > 0 for value in direct_by_phase.values())


| source | phase | voltage magnitude | unit |
| --- | --- | --- | --- |
| direct OpenDSS | 1 | 0.9997586726902581 | pu |
| direct OpenDSS | 2 | 0.9997586726902756 | pu |
| direct OpenDSS | 3 | 0.9997586726902107 | pu |


In [4]:
RUN_DIR = WORKSPACE / 'runs' / '01-first-circuit'
run_summary = cli('study', 'run', CASE_PATH, '--out', RUN_DIR, '--force')
verify_summary = cli('study', 'verify', RUN_DIR)
results = read(RUN_DIR / 'results.json')
cept_rows = [row for row in results['load_flow']['bus_voltages'] if row['bus'].lower() == 'load']
cept_by_phase = {row['phase']: row['v_pu'] for row in cept_rows}
table(['source', 'phase', 'voltage magnitude', 'unit'], [('CEPT results.json', row['phase'], row['v_pu'], 'pu') for row in cept_rows])
table(['phase', 'direct OpenDSS pu', 'CEPT pu', 'absolute difference pu'], [(phase, direct_by_phase[phase], cept_by_phase[phase], abs(direct_by_phase[phase] - cept_by_phase[phase])) for phase in (1, 2, 3)])

max_abs_diff_pu = max(abs(direct_by_phase[phase] - cept_by_phase[phase]) for phase in (1, 2, 3))
cards([
    ('CEPT run', run_summary['status'], 'solver-backed run artifacts'),
    ('Verify', str(verify_summary['passed']), 'receipt integrity and convergence'),
    ('Max |direct − CEPT|', f'{max_abs_diff_pu:.2e} pu', 'load-bus voltage agreement'),
], title='1 · Load-flow agreement')
assert run_summary['status'] == 'PASS'
assert verify_summary['passed'] is True
assert set(cept_by_phase) == {1, 2, 3}
assert max(abs(direct_by_phase[phase] - cept_by_phase[phase]) for phase in (1, 2, 3)) < 1e-4


$ cept study run '<notebook-workspace>\first_circuit_case.json' --out '<notebook-workspace>\runs\01-first-circuit' --force


→ exit 0


$ cept study verify '<notebook-workspace>\runs\01-first-circuit'


→ exit 0


| source | phase | voltage magnitude | unit |
| --- | --- | --- | --- |
| CEPT results.json | 1 | 0.999787 | pu |
| CEPT results.json | 2 | 0.999787 | pu |
| CEPT results.json | 3 | 0.999787 | pu |
| phase | direct OpenDSS pu | CEPT pu | absolute difference pu |
| --- | --- | --- | --- |
| 1 | 0.9997586726902581 | 0.999787 | 2.8327309741893458e-05 |
| 2 | 0.9997586726902756 | 0.999787 | 2.8327309724351935e-05 |
| 3 | 0.9997586726902107 | 0.999787 | 2.832730978929998e-05 |


The tables are generated from solver-returned direct values and the exact CEPT `results.json`; no output value is stored in the notebook. The verification receipt proves this public workflow's identity, convergence, finite quantities, and artifact integrity. It does not prove a real network or protection/project acceptance.